In [56]:
import import_ipynb
import EDA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [57]:
df = EDA.df

from sklearn.model_selection import train_test_split

# Step 1: Split features and target
X = df.drop("target", axis=1)  # All columns except target
y = df["target"]               # Target column




# first model Logistic uses RFE

In [62]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=9000)
rfe = RFE(model, n_features_to_select=5)
fit = rfe.fit(X, y)


selected_features = X.columns[fit.support_]
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


# Step 1: Split features and target
Xr = df[selected_features]  # All columns except target
yr = df["target"]               # Target column

# Step 2: Split into train and test sets
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2,  stratify=y, random_state=42 )

model_selected = LogisticRegression(max_iter=9000)
model_selected.fit(Xr_train, yr_train)
yr_pred = model_selected.predict(Xr_test)

print("Accuracy:", accuracy_score(yr_test, yr_pred))
print("Classification Report:\n", classification_report(yr_test, yr_pred))


Accuracy: 0.9473684210526315
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



# second model (tree based ) uses rf.feature_importances_

In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd


X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(7).index.tolist()
from sklearn.model_selection import train_test_split

# Step 1: Split features and target
Xf = df[top_features]  # All columns except target
yf = df["target"]               # Target column

# Step 2: Split into train and test sets
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,  stratify=y, random_state=42 )
rf_new = RandomForestClassifier(random_state=42)
rf_new.fit(Xf_train, yf_train)

yf_pred = rf_new.predict(Xf_test)

print("Accuracy:", accuracy_score(yf_test, yf_pred))
print("Classification Report:\n", classification_report(yf_test, yf_pred))



Accuracy: 0.9473684210526315
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



# third model uses Select k best 

In [61]:
from sklearn.feature_selection import SelectKBest, f_classif



# Step 2: Select top k features (e.g., top 5)
selector = SelectKBest(score_func=f_classif, k=5)
X_new = selector.fit_transform(X, y)
selected_mask = selector.get_support()          # Boolean mask of selected features
selected_features_ = X.columns[selected_mask]    # Column names of selected features


Xk = df[selected_features_2]  # All columns except target
yk = df["target"] 
Xk_train, Xk_test, yk_train, yk_test = train_test_split(Xk, yk, test_size=0.2,  stratify=y, random_state=42 )
rk_new = RandomForestClassifier(random_state=42)
rk_new.fit(Xk_train, yk_train)

yk_pred = rk_new.predict(Xk_test)

print("Accuracy:", accuracy_score(yk_test, yk_pred))
print("Classification Report:\n", classification_report(yk_test, yk_pred))


Accuracy: 0.9385964912280702
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.93      0.92        42
           1       0.96      0.94      0.95        72

    accuracy                           0.94       114
   macro avg       0.93      0.94      0.93       114
weighted avg       0.94      0.94      0.94       114



In [91]:
selected_features = list(selected_features)
top_features = list(top_features)
selected_features_2 = list(selected_features_2)
feature_set_1 = [selected_features]  # e.g., from RFE
feature_set_2 = [top_features]  # e.g., from RF feature importance
feature_set_3 = [selected_features_2]  # e.g., from SelectKBest
feature_set_1 = feature_set_1[0]
feature_set_2 = feature_set_2[0]
feature_set_3 = feature_set_3[0]


In [92]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define a function to train and evaluate
def train_evaluate(X, y, model=None):
    if model is None:
        model = LogisticRegression(max_iter=9000, random_state=42)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    return results

# Your three feature lists
# If your feature sets are Pandas Index objects, convert them to list


# Now your loop should work
for i, features in enumerate([feature_set_1, feature_set_2, feature_set_3], start=1):
    print(f"\nTraining model with Feature Set {i} (features count: {len(features)})")
    X = df[features]
    results = train_evaluate(X, y)
    performance[f'Set_{i}'] = results
    print(results)



# Target variable
y = df['target']

# Dictionary to store results
performance = {}

for i, features in enumerate([feature_set_1, feature_set_2, feature_set_3], start=1):
    print(f"\nTraining model with Feature Set {i} (features count: {len(features)})")
    X = df[features]
    results = train_evaluate(X, y)
    performance[f'Set_{i}'] = results
    print(results)

# Summary
print("\n=== Summary of all feature sets ===")
for set_name, metrics in performance.items():
    print(f"{set_name}: {metrics}")



Training model with Feature Set 1 (features count: 5)
{'accuracy': 0.9473684210526315, 'precision': 0.9459459459459459, 'recall': 0.9722222222222222, 'f1_score': 0.958904109589041, 'roc_auc': 0.9861111111111112}

Training model with Feature Set 2 (features count: 7)
{'accuracy': 0.9298245614035088, 'precision': 0.9444444444444444, 'recall': 0.9444444444444444, 'f1_score': 0.9444444444444444, 'roc_auc': 0.984457671957672}

Training model with Feature Set 3 (features count: 5)
{'accuracy': 0.9210526315789473, 'precision': 0.9436619718309859, 'recall': 0.9305555555555556, 'f1_score': 0.9370629370629371, 'roc_auc': 0.9821428571428571}

Training model with Feature Set 1 (features count: 5)
{'accuracy': 0.9473684210526315, 'precision': 0.9459459459459459, 'recall': 0.9722222222222222, 'f1_score': 0.958904109589041, 'roc_auc': 0.9861111111111112}

Training model with Feature Set 2 (features count: 7)
{'accuracy': 0.9298245614035088, 'precision': 0.9444444444444444, 'recall': 0.94444444444444

In [97]:

with open('selected_features.pkl', 'wb') as f:
    pickle.dump(top_features, f)


In [95]:
import pickle

# Save the model to a file
with open('rfe_model.pkl', 'wb') as file:
    pickle.dump(rf_new, file)
    
